In [1]:
!pip install kaggle

## Download dataset from kaggle and unzip if required

In [2]:
import kaggle

In [3]:
!kaggle datasets download parulverma85/netflix-titles -f netflix_titles.csv

Dataset URL: https://www.kaggle.com/datasets/parulverma85/netflix-titles
License(s): unknown




  0%|          | 0.00/3.24M [00:00<?, ?B/s]
100%|##########| 3.24M/3.24M [00:00<00:00, 649MB/s]


## Read CSV file

In [4]:
import pandas as pd
df = pd.read_csv('netflix_titles.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


## Load environment variables

In [5]:
import os
import dotenv
dotenv.load_dotenv()
mssql_db = os.getenv('mssql')

## Load netflix data to database

In [7]:
from sqlalchemy import create_engine
engine = create_engine(mssql_db)
conn = engine.connect()
df.to_sql('netflix_raw', con=conn, index=False, if_exists='append')
conn.close()

## Additional data analysis in Python

### Check the title

In [8]:
df[df['show_id']=='s5023']

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
5022,s5023,Movie,반드시 잡는다,Hong-seon Kim,Baek Yoon-sik,South Korea,"February 28, 2018",2017,TV-MA,110 min,"Dramas, International Movies, Thrillers",After people in his town start turning up dead...


### Fetch the max data length in each column

In [9]:
df.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='object')

In [10]:
df['rating'].str.len().max()

8.0

In [11]:
for col in df.columns:
    if df[col].dtype != 'int64':
        print(f'{col} : {df[col].str.len().max()}')

show_id : 5
type : 7
title : 104
director : 208.0
cast : 771.0
country : 123.0
date_added : 19.0
rating : 8.0
duration : 10.0
listed_in : 79
description : 248


### Check the null values in each columns

In [12]:
df.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64